In [51]:
!pip install sqlalchemy psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable


In [52]:
import pandas as pd
from sqlalchemy import create_engine

## Exract in E in ETL

In [ ]:

# Extract cleaned data
df = pd.read_parquet(
    r'C:\Users\harsh\OneDrive\Desktop\data Engineering\Midterm-Project\data\cleaned.parquet'
)

print("Data Loaded:", df.shape)
df.head()

Data Loaded: (2814366, 25)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration,pickup_hour,pickup_day,pickup_month,is_weekend
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,1.0,18.00,2.5,0.0,0.0,8.350000,0,2,1,0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,1.0,12.12,2.5,0.0,0.0,2.550000,0,2,1,0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,1.0,12.10,2.5,0.0,0.0,1.950000,0,2,1,0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,1.0,9.70,0.0,0.0,0.0,5.566667,0,2,1,0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,1.0,8.30,0.0,0.0,0.0,3.533333,0,2,1,0


In [54]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'trip_duration', 'pickup_hour', 'pickup_day',
       'pickup_month', 'is_weekend'],
      dtype='object')

In [55]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])

In [56]:
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.dayofweek
df["is_weekend"] = df["pickup_day"].isin([5, 6]).astype(int)

## Transform (T in ETL)

In [ ]:

# Create hourly demand dataset
hourly_demand = (
    df.groupby(
        ["pickup_date", "pickup_hour", "pickup_day", "is_weekend"]
    )
    .size()
    .reset_index(name="ride_count")
)

print("Hourly demand shape:", hourly_demand.shape)
hourly_demand.head()

Hourly demand shape: (748, 5)


,pickup_date,pickup_hour,pickup_day,is_weekend,ride_count
0,2024-12-31,20,1,0,3
1,2024-12-31,21,1,0,3
2,2024-12-31,23,1,0,15
3,2025-01-01,0,2,0,5675
4,2025-01-01,1,2,0,5509


## Load to Database (L in ETL)

In [ ]:

engine = create_engine("sqlite:///nyc_rides.db")

hourly_demand.to_sql(
    "hourly_demand",
    engine,
    if_exists="replace",
    index=False
)

print("✅ Data successfully loaded into database!")

✅ Data successfully loaded into database!


### verify data in database

In [ ]:

pd.read_sql("SELECT * FROM hourly_demand LIMIT 5", engine)

,pickup_date,pickup_hour,pickup_day,is_weekend,ride_count
0,2024-12-31,20,1,0,3
1,2024-12-31,21,1,0,3
2,2024-12-31,23,1,0,15
3,2025-01-01,0,2,0,5675
4,2025-01-01,1,2,0,5509
